# Recover a simulated galaxy with the real-source pipeline

This uses **the same `direct.prepare`, `direct.fit` and `direct.sample` functions as `direct_source_fit.ipynb`**. The fitter receives R/C observations and explicit settings; it is not given the injected emission map. This tests the current direct model, including its unknown positive brightness pixels. It does not change the real-source model.

First stage: simple 81 × 81 data, known center and noise, inclination prior 60° ± 5°, and both R registration offsets fitted. No PSF, intrinsic dispersion or correlated noise yet. A single successful recovery does not establish uncertainty coverage.

Use the **grism** kernel. **Run All starts a fresh fit when `OVERRIDE=True`.** To inspect the completed saved run without refitting, run the setup cell and the final results cell only.

**Recorded baseline result (2026-10-27): recovery failed.** The injected velocity was 200 km/s; the full real-pipeline run accumulated near 997 km/s and failed convergence. This notebook is a diagnostic, not a validated mass-measurement procedure. See [results and conclusions](docs/direct_mock_recovery_2026-10-27.md).


In [ ]:
from pathlib import Path
import os, sys, logging

# Same WSL/NVIDIA workaround as the real-source notebook, before JAX starts.
_xla_flags = [flag for flag in os.environ.get("XLA_FLAGS", "").split()
              if not flag.startswith("--xla_gpu_enable_command_buffer=")]
os.environ["XLA_FLAGS"] = " ".join([*_xla_flags, "--xla_gpu_enable_command_buffer="])
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

import torch
import matplotlib.pyplot as plt
from IPython.display import display

REPO = Path.cwd() if (Path.cwd() / "dingo").is_dir() else Path("/home/aakavan/kinematics/grism-kinematics")
sys.path.insert(0, str(REPO))
os.environ["GRISM_CAL_DIR"] = str(REPO.parent / "grism_cal")
torch.set_num_threads(min(4, os.cpu_count() or 1))
logging.getLogger("dingo.fitting").setLevel(logging.WARNING)
from dingo import simple_simulation as sim, simulation_review as inputs
from dingo import direct_workflow as direct, direct_recovery as recovery

EXPERIMENT = REPO / "results/simulations/disk_200_joint_i60s5"
RUN_NAME = "unknown_map_i60s5_start350"   # Change to preserve another fit
RUN_DIRECTORY = EXPERIMENT / "runs" / "9000001" / RUN_NAME
OVERRIDE = True                        # Replace this named fit; preserve simulated inputs
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Profile-fit device:", DEVICE)
print("Results:", RUN_DIRECTORY)

## 1. Simulated observations

An existing experiment is reopened without changing its data. To generate a different galaxy, change `EXPERIMENT` to a new folder and edit the injected values below. The source brightness image is shown only as a reference; it does not enter recovery inference.

In [ ]:
if (EXPERIMENT / "truth.json").exists():
    mock = sim.load_mock(EXPERIMENT)
    print("Reusing the saved simulated observations:", mock.directory)
else:
    mock = sim.make_mock(
        EXPERIMENT,
        velocity=200.0, turnover=5.0, inclination=60.0, pa=35.0,
        center=(40.3, 40.7), effective_radius=6.0, peak=1.0,
        noise_sigma=0.02, seed=42,
    )
fig = inputs.show_inputs(mock)
display(fig)
plt.close(fig)

## 2. Fit R+C from explicit starting guesses

These are **fitting settings**, separate from the injected truth. Velocity starts at 350 km/s, turnover radius at 3 pixels and orientation at 20°. Inclination is sampled under a Gaussian prior; the center remains fixed for this first controlled test. Both R offsets are free (`fixed_rdy=None`); C defines the reference frame.

The initial optimizer fits the shared emission pixels from R+C. Its result only initializes the MCMC; it is not a posterior uncertainty estimate.

In [ ]:
run = direct.prepare(
    mock.source_id,
    data_dir=mock.data_dir,
    output_root=EXPERIMENT / "runs",
    run_name=RUN_NAME,
    center=(40.3, 40.7),
    inclination_mean=60.0,
    inclination_sigma=5.0,
    start_inclination=60.0,
    start_velocity=350.0,
    start_turnover=3.0,
    start_pa=20.0,
    start_offset=(0.2, -0.2),
    fixed_rdy=None,
    noise_sigma=0.02,
    max_velocity=1000.0,
    device=DEVICE,
    overwrite=OVERRIDE,
)
fit_result = direct.fit(run, max_evaluations=150, progress=True)

## 3. Run the same joint MCMC as for real sources

NumPyro NUTS samples kinematics and the unknown emission map. It uses the same default brightness prior as the real-source pipeline. No injected map or true velocity is supplied here. The sampler prints its actual CPU/GPU backend, displays progress and saves checkpoints. These settings are a starting point; inspect convergence rather than assuming the step count guarantees it.

In [ ]:
posterior_path = direct.sample(
    run,
    warmup=500,
    draws=1000,
    chains=4,
    emission_prior_scale=None,
    target_accept=0.9,
    max_tree_depth=8,
    checkpoint_every=100,
    seed=42,
    device="auto",
    progress=True,
    overwrite=OVERRIDE,
    resume=False,
)

## 4. Inspect saved results and compare with truth

This block performs no fitting or sampling. After a restart, run just the setup cell and this cell to inspect the saved run. All chains enter the summaries. The final comparison shows injected truth in orange and recovered 68%/95% intervals in black; it reports truth inclusion separately from convergence. Files save inside `RUN_DIRECTORY`.

In [ ]:
views = direct.show_posterior(RUN_DIRECTORY)
display(views["summary"])
for name in ("corner", "traces", "residuals", "emission", "rotation_curve"):
    display(views[name])
    plt.close(views[name])

comparison = recovery.compare_recovery(EXPERIMENT, RUN_DIRECTORY)
display(comparison["summary"])
display(comparison["comparison"])
plt.close(comparison["comparison"])